# DATA-445: In-Class Assignment 20
---

## 1. Customer Segmentation

Consider the `customers.csv` datafile. This file contains information related to customers' activity on a company website. Below are the description of the variables.

- **ID:** customer ID
- **Visit_Time:** The number of visits to the company's website in a given month.
- **Average_Expense:** The average amount of money that the customer has spend.
- **Sex:** gender of the customer (0: female, 1: male).
- **Age:** age of the customer.

**In Python**, answer the following:

### (a) (3 points) 
Using the `pandas` library, read the csv file and create a data-frame called `customers`.

In [7]:
# %pip install --upgrade numpy
%pip install numpy==1.21.0

  Using cached numpy-1.21.0.zip (10.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'


ERROR: Exception:
Traceback (most recent call last):
  File "c:\Users\15154\anaconda3\Lib\site-packages\pip\_internal\cli\base_command.py", line 105, in _run_wrapper
    status = _inner_run()
             ^^^^^^^^^^^^
  File "c:\Users\15154\anaconda3\Lib\site-packages\pip\_internal\cli\base_command.py", line 96, in _inner_run
    return self.run(options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\15154\anaconda3\Lib\site-packages\pip\_internal\cli\req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\15154\anaconda3\Lib\site-packages\pip\_internal\commands\install.py", line 379, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "c:\Users\15154\anaconda3\Lib\site-packages\pip\_internal\resolution\resolvelib\resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "c:\Users\15154\an

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
# %pip install numpy==1.26.4
print(np.__version__)

1.26.4


In [2]:
import pandas as pd

customers = pd.read_csv('customers.csv')
customers.head()

,ID,Visit_Time,Average_Expense,Sex,Age
0,1,3,5.7,0,10
1,2,5,14.5,0,27
2,3,16,33.5,0,32
3,4,5,15.9,0,30
4,5,16,24.9,0,23


### (b) (3 points) 
Using the appropriate Python commands, remove the `ID` variable.

In [3]:
customers = customers.drop(columns=['ID'])
customers.head()

,Visit_Time,Average_Expense,Sex,Age
0,3,5.7,0,10
1,5,14.5,0,27
2,16,33.5,0,32
3,5,15.9,0,30
4,16,24.9,0,23


### (c) (8 points) 
Instead of using the principal components to visualize potential groupings, apply $t$-SNE to the (scaled) data and project it onto two dimensions. Visualize the resulting embedding with a scatterplot. Based on this visualization, how many groups (clusters) would you estimate are present in this dataset?

In [4]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.manifold import TSNE

scaler = MinMaxScaler()
customers_scaled = scaler.fit_transform(customers)

tsne = TSNE(n_components=2, perplexity=20, learning_rate='auto', init="pca", random_state=42)
customers_tsne = tsne.fit_transform(customers_scaled)

import matplotlib.pyplot as plt
plt.scatter(customers_tsne[:, 0], customers_tsne[:, 1])
plt.title('t-SNE visualization of customers')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.show()



c:\Users\15154\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=2.0.0 and <2.8.0 is required for this version of SciPy (detected version 1.26.4)
  from scipy.sparse import csr_array, issparse
c:\Users\15154\anaconda3\Lib\site-packages\scipy\sparse\_sputils.py:17: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  np.uintc, np.long, np.ulong, np.longlong, np.ulonglong,


AttributeError: module 'numpy' has no attribute 'long'

Based on the above chart, t-SNE is suggesting that there are four natural groups in the data.

## 2. Heart Disease Classification: UMAP for Feature Engineering

Consider the `framingham.csv` data file described in previous in class assignments. Recall that the classification goal is to predict whether a patient has a 10-year risk of future coronary heart disease (`TenYearCHD`).

In **Python**, answer the following:

### (a) (4 points) 
Using the `pandas` library, read the csv data file and create a data-frame called `heart`. Remove the missing values.

In [5]:
heart = pd.read_csv('framingham.csv')
heart = heart.dropna().reset_index(drop=True)
heart.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


### (b) (10 points) 
Using `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the predictor variables, and `TenYearCHD` as the target variable, set up a `StratifiedKFold` with 10 folds (taking into account the proportion of 0s and 1s). Using the original predictors (no feature engineering), train a `LogisticRegression` model and use the area under the ROC-AUC curve as a measure of performance. Make sure to use the `cross_val_score` function with your `StratifiedKFold` object. This is your **baseline** out-of-sample performance.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

X = heart[['age', 'currentSmoker', 'totChol', 'BMI', 'heartRate']]
y = heart['TenYearCHD']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logic_mdl = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

logic_scores = cross_val_score(logic_mdl, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)
print(f'Logistic Regression ROC AUC scores: {logic_scores}')



c:\Users\15154\anaconda3\Lib\site-packages\scipy\sparse\_sputils.py:17: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  np.uintc, np.long, np.ulong, np.longlong, np.ulonglong,


AttributeError: module 'numpy' has no attribute 'long'

### (c) (12 points) 
Use `UMAP` as a feature engineering step: build a `Pipeline` that first standardizes the predictor variables (e.g., using `StandardScaler`), then applies `UMAP` to the standardized variables (e.g., 2 components), and then trains a `LogisticRegression` model on the resulting embedding. Using the **same** `StratifiedKFold` object from part (b), compute the cross-validated area under the ROC-AUC curve with `cross_val_score`. Fitting `StandardScaler` and `UMAP` inside the pipeline ensures that both are re-fit on only the training fold each time, so the test fold remains truly out-of-sample.

In [2]:
%pip install umap-learn
# %pip install --upgrade numpy


  Using cached numpy-2.0.2-cp312-cp312-win_amd64.whl.metadata (59 kB)
Using cached numpy-2.0.2-cp312-cp312-win_amd64.whl (15.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.9.1 which is incompatible.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.0.2 which is incompatible.
fuzzy-c-means 2.3.1 requires numpy<2,>=1.21.1, but you have numpy 2.0.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.18.1 which is incompatible.
matplotlib 3.7.5 requires numpy<2,>=1.20, but you have numpy 2.0.2 which is incompatible.
pandas 2.1.4 requires numpy<2,>=1.26.0; python_version >= "3.12", but you have numpy 2.0.2 which is incompatible.
pycaret 3.3.2 requires joblib<1.4,>=1.2.0, but you have joblib

In [7]:
# %pip install umap-learn
from umap import UMAP

umap_md = Pipeline([
    ('scaler', StandardScaler()),
    ('umap', UMAP(n_components=2, random_state=42))
])

umap_scores = cross_val_score(umap_md, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)
print(f'UMAP ROC AUC scores: {umap_scores}')


c:\Users\15154\anaconda3\Lib\site-packages\scipy\sparse\_sputils.py:17: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  np.uintc, np.long, np.ulong, np.longlong, np.ulonglong,


AttributeError: module 'numpy' has no attribute 'long'

### (d) (4 points) 
Compare the (mean) cross-validated area under the ROC-AUC curve obtained in parts (b) and (c). Based on this comparison, did using `UMAP` as a feature engineering step improve, worsen, or have no meaningful effect on the model's out-of-sample performance?

_Your answer here._